# ============================================
# 01_uci_household_power_downloader.ipynb
# UCI Individual Household Electric Power Consumption
# ============================================

## 📌 Dataset Title
UCI Individual Household Electric Power Consumption

## 🌐 Source URL
Official dataset page:  
https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption

Direct download URL:  
https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip

## 🎯 Purpose of This Dataset
This dataset is used as the **first real energy time-series dataset** in the PhD pipeline.

It is useful for:
- data preprocessing
- missing-value handling
- timestamp parsing
- resampling
- time-series feature engineering
- early forecasting baselines

Although it is household rather than factory data, it is an excellent **starter dataset** because it is:
- public
- reliable
- structured
- widely used in time-series energy research

## ❓ Why We Need This Dataset
We need this dataset at the beginning because it helps us build and test the **complete preprocessing pipeline** before moving to more complex industrial datasets.

This allows us to verify:
- input data loading
- cleaning workflow
- date-time indexing
- rolling windows
- feature extraction
- storage to Google Drive

## ⚙️ What We Are Doing in This Notebook
In this notebook, we will:

1. Mount Google Drive
2. Create project folders
3. Download the dataset ZIP from the official URL
4. Extract the files
5. Identify CSV/TXT/tabular files
6. Save a manifest
7. Repackage the extracted usable files into a ZIP bundle
8. Store everything in Google Drive

## 📂 Where This Data Will Be Used Next
This dataset will be used in the following next notebooks:

- **Notebook 1: Data Preprocessing**
- **Notebook 2: Baseline Models**
- optionally for early tests in **Notebook 3: NEO-Forecast**

## 📦 Expected Outputs
At the end of this notebook, we will have:

- raw downloaded ZIP
- extracted text/data files
- dataset manifest JSON
- repackaged ZIP bundle of usable files

## ✅ Result
This notebook establishes the first reproducible real-data acquisition step in the thesis workflow.

In [1]:
# ============================================
# 01_uci_household_power_downloader.ipynb
# Python code
# ============================================

import os
import json
import shutil
import zipfile
from pathlib import Path
from urllib.parse import urlparse

import requests

# -----------------------------
# 1. Dataset metadata
# -----------------------------
DATASET_NAME = "UCI Individual Household Electric Power Consumption"
SOURCE_PAGE = "https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption"
DOWNLOAD_URL = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"

print("Dataset Name :", DATASET_NAME)
print("Source Page  :", SOURCE_PAGE)
print("Download URL :", DOWNLOAD_URL)

# -----------------------------
# 2. Mount Google Drive
# -----------------------------
from google.colab import drive
drive.mount('/content/drive')

# -----------------------------
# 3. Create folder structure
# -----------------------------
project_root = Path("/content/drive/MyDrive/PhD_Thesis_Datasets")
dataset_root = project_root / "01_uci_household_power"
raw_dir = dataset_root / "raw"
extracted_dir = dataset_root / "extracted"
bundle_dir = dataset_root / "bundle"

for folder in [project_root, dataset_root, raw_dir, extracted_dir, bundle_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("\nCreated folders:")
print(" -", project_root)
print(" -", dataset_root)
print(" -", raw_dir)
print(" -", extracted_dir)
print(" -", bundle_dir)

# -----------------------------
# 4. Download helper
# -----------------------------
def download_file(url: str, dest_path: Path, chunk_size: int = 1024 * 1024):
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return dest_path

# -----------------------------
# 5. Download dataset ZIP
# -----------------------------
filename = os.path.basename(urlparse(DOWNLOAD_URL).path)
download_path = raw_dir / filename

if not download_path.exists():
    print("\nDownloading dataset...")
    download_file(DOWNLOAD_URL, download_path)
    print("Download complete:", download_path)
else:
    print("\nUsing cached file:", download_path)

# -----------------------------
# 6. Extract ZIP
# -----------------------------
print("\nExtracting ZIP...")
with zipfile.ZipFile(download_path, "r") as zf:
    zf.extractall(extracted_dir)

print("Extraction complete:", extracted_dir)

# -----------------------------
# 7. List extracted files
# -----------------------------
all_files = sorted([p for p in extracted_dir.rglob("*") if p.is_file()])
usable_files = [
    p for p in all_files
    if p.suffix.lower() in [".txt", ".csv", ".data", ".json", ".xlsx", ".xls"]
]

print("\nTotal extracted files:", len(all_files))
print("Usable tabular/text files:", len(usable_files))
for fp in usable_files:
    print(" -", fp)

# -----------------------------
# 8. Save dataset manifest
# -----------------------------
manifest = {
    "dataset_name": DATASET_NAME,
    "source_page": SOURCE_PAGE,
    "download_url": DOWNLOAD_URL,
    "downloaded_file": str(download_path),
    "extracted_dir": str(extracted_dir),
    "all_files": [str(p) for p in all_files],
    "usable_files": [str(p) for p in usable_files],
    "used_in_next_notebooks": [
        "Notebook 1: Data Preprocessing",
        "Notebook 2: Baseline Models",
        "Notebook 3: NEO-Forecast (early testing)"
    ]
}

manifest_path = dataset_root / "dataset_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved manifest:", manifest_path)

# -----------------------------
# 9. Repackage usable files into ZIP
# -----------------------------
bundle_zip_path = bundle_dir / "uci_household_power_usable_files.zip"

with zipfile.ZipFile(bundle_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in usable_files:
        arcname = fp.relative_to(extracted_dir)
        zf.write(fp, arcname)

print("Created bundle ZIP:", bundle_zip_path)

# -----------------------------
# 10. Optional local runtime copy
# -----------------------------
runtime_copy = Path("/content/uci_household_power_manifest_copy.json")
shutil.copy2(manifest_path, runtime_copy)
print("Runtime copy created:", runtime_copy)

# -----------------------------
# 11. Summary
# -----------------------------
print("\n===== SUMMARY =====")
print("Dataset root      :", dataset_root)
print("Raw ZIP           :", download_path)
print("Extracted folder  :", extracted_dir)
print("Manifest          :", manifest_path)
print("Bundle ZIP        :", bundle_zip_path)
print("===================")

Dataset Name : UCI Individual Household Electric Power Consumption
Source Page  : https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption
Download URL : https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Created folders:
 - /content/drive/MyDrive/PhD_Thesis_Datasets
 - /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power
 - /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/raw
 - /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/extracted
 - /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/bundle

Using cached file: /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/raw/individual+household+electric+power+consumption.zip

Extracting ZIP...
Extraction complete: /content/drive/MyDrive/PhD_Thesis_Datasets/01

In [2]:
# ============================================
# FINAL STEP: Download to PC + Create Full ZIP
# ============================================

from google.colab import files
import zipfile
from pathlib import Path

print("Preparing files for download...")

# -----------------------------
# 1. Create FULL dataset ZIP (best practice)
# -----------------------------
full_zip_path = dataset_root / "uci_household_power_FULL_PACKAGE.zip"

with zipfile.ZipFile(full_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:

    # Add raw ZIP
    if download_path.exists():
        zf.write(download_path, arcname="raw/" + download_path.name)

    # Add extracted files
    for file in extracted_dir.rglob("*"):
        if file.is_file():
            zf.write(file, arcname="extracted/" + str(file.relative_to(extracted_dir)))

    # Add bundle ZIP
    if bundle_zip_path.exists():
        zf.write(bundle_zip_path, arcname="bundle/" + bundle_zip_path.name)

    # Add manifest
    if manifest_path.exists():
        zf.write(manifest_path, arcname="manifest/" + manifest_path.name)

print("Full package created:")
print(full_zip_path)

# -----------------------------
# 2. Download key files to PC
# -----------------------------
print("\nStarting downloads to your PC...")

try:
    # Manifest
    files.download(str(manifest_path))

    # Clean bundle (most useful)
    files.download(str(bundle_zip_path))

    # Full package (recommended)
    files.download(str(full_zip_path))

    print("Download started successfully!")

except Exception as e:
    print("Download error:", e)

# -----------------------------
# 3. Confirm Drive storage
# -----------------------------
print("\nFiles stored permanently in Google Drive:")
print(" - Raw ZIP:", download_path)
print(" - Extracted:", extracted_dir)
print(" - Bundle:", bundle_zip_path)
print(" - Full Package:", full_zip_path)

Preparing files for download...
Full package created:
/content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/uci_household_power_FULL_PACKAGE.zip

Starting downloads to your PC...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started successfully!

Files stored permanently in Google Drive:
 - Raw ZIP: /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/raw/individual+household+electric+power+consumption.zip
 - Extracted: /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/extracted
 - Bundle: /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/bundle/uci_household_power_usable_files.zip
 - Full Package: /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/uci_household_power_FULL_PACKAGE.zip


In [3]:
# ============================================
# FINAL COMPLETE CELL: SAVE + PACKAGE + DOWNLOAD
# ============================================

from google.colab import files
from pathlib import Path
import zipfile
import os

print("===== FINAL EXPORT STARTED =====\n")

# -----------------------------
# 1. Verify files exist
# -----------------------------
assert download_path.exists(), "Raw ZIP missing!"
assert extracted_dir.exists(), "Extracted folder missing!"
assert manifest_path.exists(), "Manifest missing!"
assert bundle_zip_path.exists(), "Bundle ZIP missing!"

print("All required files exist ✅\n")

# -----------------------------
# 2. Create FULL PACKAGE in Google Drive
# -----------------------------
full_zip_path = dataset_root / "uci_household_power_FULL_PACKAGE.zip"

with zipfile.ZipFile(full_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:

    # Raw ZIP
    zf.write(download_path, arcname="raw/" + download_path.name)

    # Extracted files
    for file in extracted_dir.rglob("*"):
        if file.is_file():
            zf.write(file, arcname="extracted/" + str(file.relative_to(extracted_dir)))

    # Clean bundle
    zf.write(bundle_zip_path, arcname="bundle/" + bundle_zip_path.name)

    # Manifest
    zf.write(manifest_path, arcname="manifest/" + manifest_path.name)

print("Full package created in Google Drive:")
print(full_zip_path)

# -----------------------------
# 3. Verify Google Drive storage
# -----------------------------
print("\n===== GOOGLE DRIVE CONTENT =====")
for p in dataset_root.rglob("*"):
    print(p)

# -----------------------------
# 4. Download to your PC
# -----------------------------
print("\nStarting download to your PC...")

files.download(str(full_zip_path))

print("\nDownload initiated ✅")

# -----------------------------
# 5. Final summary
# -----------------------------
print("\n===== FINAL SUMMARY =====")
print("Google Drive Location:")
print(dataset_root)

print("\nKey Files:")
print("Raw ZIP        :", download_path)
print("Extracted Data :", extracted_dir)
print("Manifest       :", manifest_path)
print("Bundle ZIP     :", bundle_zip_path)
print("Full Package   :", full_zip_path)

print("\n===== DONE SUCCESSFULLY =====")

===== FINAL EXPORT STARTED =====

All required files exist ✅

Full package created in Google Drive:
/content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/uci_household_power_FULL_PACKAGE.zip

===== GOOGLE DRIVE CONTENT =====
/content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/raw
/content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/extracted
/content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/bundle
/content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/dataset_manifest.json
/content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/uci_household_power_FULL_PACKAGE.zip
/content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/raw/individual+household+electric+power+consumption.zip
/content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/raw/household_power.zip
/content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/raw/household_power_consumption.txt
/content/drive/MyDrive/PhD_Thesis_Datas

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download initiated ✅

===== FINAL SUMMARY =====
Google Drive Location:
/content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power

Key Files:
Raw ZIP        : /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/raw/individual+household+electric+power+consumption.zip
Extracted Data : /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/extracted
Manifest       : /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/dataset_manifest.json
Bundle ZIP     : /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/bundle/uci_household_power_usable_files.zip
Full Package   : /content/drive/MyDrive/PhD_Thesis_Datasets/01_uci_household_power/uci_household_power_FULL_PACKAGE.zip

===== DONE SUCCESSFULLY =====
